In [1]:
from pathlib import Path

import polars as pl
from morfeus import BuriedVolume, ConeAngle, Dispersion, Sterimol, XTB

import grrmlib as gl

In [2]:
gl.__version__

'0.1.1'

# 0. Featurization of MPAA

In [3]:
paths = sorted(Path("backbone").glob("*.com"))
rows = []

for path in paths:
    reader = gl.GaussianInputReader()
    mol = reader.read(path)
    xtb = XTB(
        mol.symbols,
        mol.atomcoords
    )
    ca = ConeAngle(
        mol.symbols,
        mol.atomcoords,
        atom_1=1,
        method="internal"
    )
    disp = Dispersion(
        mol.symbols,
        mol.atomcoords
    )
    row = {
        "backbone": int(path.stem),
        "backbone_homo": xtb.get_homo(),
        "backbone_lumo": xtb.get_lumo(),
        "backbone_ip": xtb.get_ip(),
        "backbone_ea": xtb.get_ea(),
        "backbone_cone_angle": ca.cone_angle
    }
    
    charges = xtb.get_charges()
    fukui_minus = xtb.get_fukui("nucleophilicity")
    fukui_zero = xtb.get_fukui("radical")
    fukui_plus = xtb.get_fukui("electrophilicity")
    polarizabilities = xtb.get_atom_polarizabilities()
    dipoles = xtb.get_atom_dipole_moments()
    p_ints = disp.atom_p_int
    
    labels = [1, 3, 7, 9]
    attached_indices = [7, 1, 1, 1]
    
    for label, attached_index in zip(labels, attached_indices):
        bv = BuriedVolume(
            mol.symbols,
            mol.atomcoords,
            metal_index=label
        )
        st = Sterimol(
            mol.symbols,
            mol.atomcoords,
            dummy_index=label,
            attached_index=attached_index
        )
        row_label = {
            f"backbone_charge_label{label}": charges[label],
            f"backbone_fukui_minus_label{label}": fukui_minus[label],
            f"backbone_fukui_zero_label{label}": fukui_zero[label],
            f"backbone_fukui_plus_label{label}": fukui_plus[label],
            f"backbone_polarizability_label{label}": polarizabilities[label],
            f"backbone_dipole_label{label}": dipoles[label],
            f"backbone_p_int_label{label}": p_ints[label],
            f"backbone_vbur_label{label}": bv.fraction_buried_volume * 100,
            f"backbone_B1_label{label}": st.B_1_value,
            f"backbone_B5_label{label}": st.B_5_value,
            f"backbone_L_label{label}": st.L_value,
        }
        row |= row_label
    
    rows.append(row)

df = pl.DataFrame(rows).sort("backbone")

In [4]:
df

backbone,backbone_homo,backbone_lumo,backbone_ip,backbone_ea,backbone_cone_angle,backbone_charge_label1,backbone_fukui_minus_label1,backbone_fukui_zero_label1,backbone_fukui_plus_label1,backbone_polarizability_label1,backbone_dipole_label1,backbone_p_int_label1,backbone_vbur_label1,backbone_B1_label1,backbone_B5_label1,backbone_L_label1,backbone_charge_label3,backbone_fukui_minus_label3,backbone_fukui_zero_label3,backbone_fukui_plus_label3,backbone_polarizability_label3,backbone_dipole_label3,backbone_p_int_label3,backbone_vbur_label3,backbone_B1_label3,backbone_B5_label3,backbone_L_label3,backbone_charge_label7,backbone_fukui_minus_label7,backbone_fukui_zero_label7,backbone_fukui_plus_label7,backbone_polarizability_label7,backbone_dipole_label7,backbone_p_int_label7,backbone_vbur_label7,backbone_B1_label7,backbone_B5_label7,backbone_L_label7,backbone_charge_label9,backbone_fukui_minus_label9,backbone_fukui_zero_label9,backbone_fukui_plus_label9,backbone_polarizability_label9,backbone_dipole_label9,backbone_p_int_label9,backbone_vbur_label9,backbone_B1_label9,backbone_B5_label9,backbone_L_label9
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,-0.388311,-0.319427,8.5047,1.0886,239.526291,0.3509,0.485,0.543,0.601,32.986,0.875182,27.074261,39.668165,1.763298,5.084882,5.775934,-0.350701,0.042,0.023,0.003,6.028,0.185557,20.089336,48.406634,2.1,5.473037,5.822896,-0.146814,-0.005,0.011,0.026,7.303,0.075374,30.874694,64.818999,2.1,5.084883,4.404579,-0.365614,0.048,0.044,0.039,6.107,0.234984,18.343721,43.766306,2.1,4.729343,6.504397
1,-0.384586,-0.317421,8.314,1.0756,239.128401,0.331844,0.458,0.523,0.588,33.036,0.901613,27.329587,39.952362,2.117552,5.077564,6.085845,-0.350211,0.043,0.023,0.002,6.027,0.185263,20.721648,49.129939,2.1,5.595524,6.066393,-0.160094,-0.014,0.005,0.025,7.346,0.09521,32.479729,70.157589,2.117132,5.077564,4.411729,-0.357954,0.042,0.04,0.039,6.086,0.229771,18.631499,44.016143,2.1,5.931428,6.483278
2,-0.381723,-0.314163,7.9751,1.0194,239.453323,0.334102,0.374,0.481,0.588,33.03,0.901504,27.795445,40.050783,2.252704,5.400756,7.650898,-0.351742,0.032,0.016,-0.0,6.031,0.184999,21.14598,49.525368,2.1,7.376626,5.946715,-0.156069,-0.02,0.002,0.025,7.333,0.07419,36.036076,70.994456,2.254314,5.400756,4.4051,-0.366347,0.035,0.035,0.035,6.108,0.233402,18.908874,44.094763,2.1,8.459991,6.488024
3,-0.381061,-0.315206,8.0633,1.0507,239.322728,0.329289,0.401,0.493,0.585,33.042,0.912037,27.824986,40.488143,2.399934,5.083051,6.572473,-0.353415,0.038,0.019,-0.001,6.035,0.185629,21.36554,50.069884,2.1,6.525664,5.779004,-0.152234,-0.009,0.009,0.027,7.321,0.086977,37.912943,74.040253,2.3998,5.083051,4.413117,-0.364245,0.047,0.042,0.036,6.099,0.232382,19.316483,44.726635,2.1,7.132493,6.395848
4,-0.380868,-0.315417,7.9883,1.0663,239.203719,0.330771,0.377,0.48,0.584,33.038,0.912371,28.011429,40.508526,2.417494,5.082186,7.375275,-0.353041,0.035,0.017,-0.001,6.034,0.185712,21.451692,50.085608,2.1,7.56085,5.767325,-0.155152,-0.012,0.008,0.027,7.33,0.087913,38.28695,74.11305,2.416217,5.082187,4.413435,-0.363592,0.044,0.04,0.036,6.097,0.232508,19.050869,44.747601,2.1,7.687006,6.387257
5,-0.385259,-0.318421,7.8419,1.2262,239.043471,0.332434,0.344,0.442,0.539,33.034,0.897965,27.751808,39.977986,1.933638,6.334368,9.66926,-0.349781,0.028,0.015,0.003,6.026,0.185061,21.251516,49.11072,2.1,9.204841,5.896157,-0.16301,-0.017,0.002,0.021,7.355,0.101169,33.842789,70.988632,2.1,6.334368,4.411608,-0.3565,0.031,0.033,0.034,6.081,0.229439,18.929277,44.093016,2.1,10.074941,6.476354
19,-0.377896,-0.312465,7.9504,0.9966,239.376028,0.312464,0.392,0.485,0.579,33.086,0.962951,28.189918,42.124022,2.452604,5.081782,6.648393,-0.356155,0.037,0.018,-0.001,6.043,0.18581,22.543124,53.15936,2.1,6.614854,5.781332,-0.149451,-0.008,0.01,0.029,7.312,0.087569,38.675613,76.719158,2.451211,5.081782,4.415139,-0.367692,0.046,0.04

In [5]:
df.write_parquet("features_backbone.parquet")

# 1. Featurization of Pyridone

In [6]:
paths = sorted(Path("pyridone").glob("*.com"))
rows = []

for path in paths:
    reader = gl.GaussianInputReader()
    mol = reader.read(path)
    xtb = XTB(
        mol.symbols,
        mol.atomcoords
    )
    disp = Dispersion(
        mol.symbols,
        mol.atomcoords
    )
    row = {
        "pyridone": int(path.stem),
        "pyridone_homo": xtb.get_homo(),
        "pyridone_lumo": xtb.get_lumo(),
        "pyridone_ip": xtb.get_ip(),
        "pyridone_ea": xtb.get_ea(),
    }
    
    charges = xtb.get_charges()
    fukui_minus = xtb.get_fukui("nucleophilicity")
    fukui_zero = xtb.get_fukui("radical")
    fukui_plus = xtb.get_fukui("electrophilicity")
    polarizabilities = xtb.get_atom_polarizabilities()
    dipoles = xtb.get_atom_dipole_moments()
    p_ints = disp.atom_p_int
    
    labels = [1, 2, 3]
    attached_indices = [4, 1, 4]
    
    for label, attached_index in zip(labels, attached_indices):
        bv = BuriedVolume(
            mol.symbols,
            mol.atomcoords,
            metal_index=label
        )
        st = Sterimol(
            mol.symbols,
            mol.atomcoords,
            dummy_index=label,
            attached_index=attached_index
        )
        row_label = {
            f"pyridone_charge_label{label}": charges[label],
            f"pyridone_fukui_minus_label{label}": fukui_minus[label],
            f"pyridone_fukui_zero_label{label}": fukui_zero[label],
            f"pyridone_fukui_plus_label{label}": fukui_plus[label],
            f"pyridone_polarizability_label{label}": polarizabilities[label],
            f"pyridone_dipole_label{label}": dipoles[label],
            f"pyridone_p_int_label{label}": p_ints[label],
            f"pyridone_vbur_label{label}": bv.fraction_buried_volume * 100,
            f"pyridone_B1_label{label}": st.B_1_value,
            f"pyridone_B5_label{label}": st.B_5_value,
            f"pyridone_L_label{label}": st.L_value,
        }
        row |= row_label
    
    rows.append(row)

df = pl.DataFrame(rows)

In [7]:
df

pyridone,pyridone_homo,pyridone_lumo,pyridone_ip,pyridone_ea,pyridone_charge_label1,pyridone_fukui_minus_label1,pyridone_fukui_zero_label1,pyridone_fukui_plus_label1,pyridone_polarizability_label1,pyridone_dipole_label1,pyridone_p_int_label1,pyridone_vbur_label1,pyridone_B1_label1,pyridone_B5_label1,pyridone_L_label1,pyridone_charge_label2,pyridone_fukui_minus_label2,pyridone_fukui_zero_label2,pyridone_fukui_plus_label2,pyridone_polarizability_label2,pyridone_dipole_label2,pyridone_p_int_label2,pyridone_vbur_label2,pyridone_B1_label2,pyridone_B5_label2,pyridone_L_label2,pyridone_charge_label3,pyridone_fukui_minus_label3,pyridone_fukui_zero_label3,pyridone_fukui_plus_label3,pyridone_polarizability_label3,pyridone_dipole_label3,pyridone_p_int_label3,pyridone_vbur_label3,pyridone_B1_label3,pyridone_B5_label3,pyridone_L_label3
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,-0.380848,-0.250706,8.9152,-1.3769,-0.379884,0.112,0.091,0.07,6.107,0.198239,14.02238,28.841898,1.700434,3.285902,6.667683,0.32495,0.054,0.051,0.048,1.267,0.08328,11.822243,29.208209,1.52,6.096487,5.617428,-0.277363,0.195,0.178,0.16,7.847,0.278937,18.253291,45.716083,1.7,4.440932,4.6939
1,-0.406456,-0.280751,8.7322,-0.1788,-0.348909,0.092,0.077,0.062,6.02,0.206616,16.245055,33.820583,1.750047,4.484329,7.96781,0.3370547,0.048,0.046,0.044,1.236,0.083711,12.320277,30.485348,1.52,7.369946,6.074914,-0.250416,0.101,0.125,0.149,7.753,0.280977,19.119695,46.2804,1.7,5.622741,5.957405
10,-0.376059,-0.272477,8.278,-0.1106,-0.378689,0.081,0.07,0.06,6.103,0.196786,14.716459,28.912947,1.700488,4.498522,8.750446,0.323833,0.039,0.039,0.038,1.27,0.083712,12.197199,29.214615,1.52,8.42273,5.697263,-0.29608,0.147,0.121,0.096,7.914,0.278272,21.104492,51.455344,1.7,5.706515,4.698209
11,-0.373141,-0.268191,7.8267,0.0904,-0.377685,0.066,0.059,0.052,6.1,0.195773,16.797702,33.248113,1.701356,5.814936,8.743697,0.3235825,0.037,0.037,0.037,1.27,0.084233,12.979299,30.470206,1.52,8.444513,8.057512,-0.303265,0.107,0.094,0.081,7.94,0.275026,21.847751,51.731387,1.7,5.71321,6.876921
12,-0.377621,-0.2411,8.2676,-1.4806,-0.368172,0.1,0.086,0.071,6.074,0.201464,15.191776,32.061242,1.77081,5.469567,6.669206,0.326579,0.053,0.05,0.048,1.263,0.08128,12.065231,30.075941,1.52,6.093346,7.062119,-0.276954,0.112,0.135,0.158,7.846,0.280493,18.47447,45.917001,1.7,4.449981,6.792953
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
5,-0.376779,-0.243526,8.5306,-1.4383,-0.381704,0.098,0.081,0.064,6.112,0.1988608,15.832849,33.471743,1.788619,4.507627,6.662146,0.321276,0.051,0.048,0.045,1.276,0.083713,12.214345,30.545914,1.52,6.089425,6.267824,-0.281013,0.154,0.152,0.15,7.86,0.276741,18.573316,45.869829,1.825551,4.43831,5.727413
6,-0.377388,-0.243575,8.7115,-1.4156,-0.382197,0.109,0.087,0.066,6.113,0.198531,14.29637,28.852963,1.788699,4.496712,6.659331,0.323633,0.052,0.049,0.046,1.27,0.083213,11.759912,29.238492,1.52,6.085145,6.711125,-0.285807,0.182,0.168,0.153,7.877,0.279299,18.145517,45.664834,1.7,5.499003,4.819833
7,-0.376674,-0.246249,8.527,-1.4109,-0.383183,0.11,0.088,0.066,6.116,0.198251,14.163164,28.77842,1.907353,3.279125,7.709017,0.323317,0.051,0.049,0.046,1.271,0.083137,11.882134,29.198891,1.52,7.145886,5.613327,-0.28046,0.156,0.155,0.154,7.858,0.277472,18.484877,46.012509,1.7,5.517145,4.682176


In [8]:
df.write_parquet("features_pyridone.parquet")

# 2. Make Dataset

In [3]:
import polars as pl

df_backbone = pl.read_parquet("features_backbone.parquet")
df_pyridone = pl.read_parquet("features_pyridone.parquet")

In [4]:
df_X = df_backbone.join(df_pyridone, how="cross")

In [5]:
df_X

backbone,backbone_homo,backbone_lumo,backbone_ip,backbone_ea,backbone_cone_angle,backbone_charge_label1,backbone_fukui_minus_label1,backbone_fukui_zero_label1,backbone_fukui_plus_label1,backbone_polarizability_label1,backbone_dipole_label1,backbone_p_int_label1,backbone_vbur_label1,backbone_B1_label1,backbone_B5_label1,backbone_L_label1,backbone_charge_label3,backbone_fukui_minus_label3,backbone_fukui_zero_label3,backbone_fukui_plus_label3,backbone_polarizability_label3,backbone_dipole_label3,backbone_p_int_label3,backbone_vbur_label3,backbone_B1_label3,backbone_B5_label3,backbone_L_label3,backbone_charge_label7,backbone_fukui_minus_label7,backbone_fukui_zero_label7,backbone_fukui_plus_label7,backbone_polarizability_label7,backbone_dipole_label7,backbone_p_int_label7,backbone_vbur_label7,backbone_B1_label7,…,pyridone_homo,pyridone_lumo,pyridone_ip,pyridone_ea,pyridone_charge_label1,pyridone_fukui_minus_label1,pyridone_fukui_zero_label1,pyridone_fukui_plus_label1,pyridone_polarizability_label1,pyridone_dipole_label1,pyridone_p_int_label1,pyridone_vbur_label1,pyridone_B1_label1,pyridone_B5_label1,pyridone_L_label1,pyridone_charge_label2,pyridone_fukui_minus_label2,pyridone_fukui_zero_label2,pyridone_fukui_plus_label2,pyridone_polarizability_label2,pyridone_dipole_label2,pyridone_p_int_label2,pyridone_vbur_label2,pyridone_B1_label2,pyridone_B5_label2,pyridone_L_label2,pyridone_charge_label3,pyridone_fukui_minus_label3,pyridone_fukui_zero_label3,pyridone_fukui_plus_label3,pyridone_polarizability_label3,pyridone_dipole_label3,pyridone_p_int_label3,pyridone_vbur_label3,pyridone_B1_label3,pyridone_B5_label3,pyridone_L_label3
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,-0.388311,-0.319427,8.5047,1.0886,239.526291,0.3509,0.485,0.543,0.601,32.986,0.875182,27.074261,39.668165,1.763298,5.084882,5.775934,-0.350701,0.042,0.023,0.003,6.028,0.185557,20.089336,48.406634,2.1,5.473037,5.822896,-0.146814,-0.005,0.011,0.026,7.303,0.075374,30.874694,64.818999,2.1,…,-0.380848,-0.250706,8.9152,-1.3769,-0.379884,0.112,0.091,0.07,6.107,0.198239,14.02238,28.841898,1.700434,3.285902,6.667683,0.32495,0.054,0.051,0.048,1.267,0.08328,11.822243,29.208209,1.52,6.096487,5.617428,-0.277363,0.195,0.178,0.16,7.847,0.278937,18.253291,45.716083,1.7,4.440932,4.6939
0,-0.388311,-0.319427,8.5047,1.0886,239.526291,0.3509,0.485,0.543,0.601,32.986,0.875182,27.074261,39.668165,1.763298,5.084882,5.775934,-0.350701,0.042,0.023,0.003,6.028,0.185557,20.089336,48.406634,2.1,5.473037,5.822896,-0.146814,-0.005,0.011,0.026,7.303,0.075374,30.874694,64.818999,2.1,…,-0.406456,-0.280751,8.7322,-0.1788,-0.348909,0.092,0.077,0.062,6.02,0.206616,16.245055,33.820583,1.750047,4.484329,7.96781,0.3370547,0.048,0.046,0.044,1.236,0.083711,12.320277,30.485348,1.52,7.369946,6.074914,-0.250416,0.101,0.125,0.149,7.753,0.280977,19.119695,46.2804,1.7,5.622741,5.957405
0,-0.388311,-0.319427,8.5047,1.0886,239.526291,0.3509,0.485,0.543,0.601,32.986,0.875182,27.074261,39.668165,1.763298,5.084882,5.775934,-0.350701,0.042,0.023,0.003,6.028,0.185557,20.089336,48.406634,2.1,5.473037,5.822896,-0.146814,-0.005,0.011,0.026,7.303,0.075374,30.874694,64.818999,2.1,…,-0.376059,-0.272477,8.278,-0.1106,-0.378689,0.081,0.07,0.06,6.103,0.196786,14.716459,28.912947,1.700488,4.498522,8.750446,0.323833,0.039,0.039,0.038,1.27,0.083712,12.197199,29.214615,1.52,8.42273,5.697263,-0.29608,0.147,0.121,0.096,7.914,0.278272,21.104492,51.455344,1.7,5.706515,4.698209
0,-0.388311,-0.319427,8.5047,1.0886,239.526291,0.3509,0.485,0.543,0.601,32.986,0.875182,27.074261,39.668165,1.763298,5.084882,5.775934,-0.350701,0.042,0.023,0.003,6.028,0.185557,20.089336,48.406634,2.1,5.473037,5.822896,-0.146814,-0.005,0.011,0.026,7.303,0.075374,30.874694,64.818999,2.1,…,-0.373141,-0.268191,7.8267,

In [6]:
columns_feat = [col for col in df_X.columns if col not in ["pyridone", "backbone"]]
df_X = (
    df_X
    .select(["backbone", "pyridone"] + columns_feat)
    .sort(["pyridone", "backbone"])
)

In [7]:
df_X

backbone,pyridone,backbone_homo,backbone_lumo,backbone_ip,backbone_ea,backbone_cone_angle,backbone_charge_label1,backbone_fukui_minus_label1,backbone_fukui_zero_label1,backbone_fukui_plus_label1,backbone_polarizability_label1,backbone_dipole_label1,backbone_p_int_label1,backbone_vbur_label1,backbone_B1_label1,backbone_B5_label1,backbone_L_label1,backbone_charge_label3,backbone_fukui_minus_label3,backbone_fukui_zero_label3,backbone_fukui_plus_label3,backbone_polarizability_label3,backbone_dipole_label3,backbone_p_int_label3,backbone_vbur_label3,backbone_B1_label3,backbone_B5_label3,backbone_L_label3,backbone_charge_label7,backbone_fukui_minus_label7,backbone_fukui_zero_label7,backbone_fukui_plus_label7,backbone_polarizability_label7,backbone_dipole_label7,backbone_p_int_label7,backbone_vbur_label7,…,pyridone_homo,pyridone_lumo,pyridone_ip,pyridone_ea,pyridone_charge_label1,pyridone_fukui_minus_label1,pyridone_fukui_zero_label1,pyridone_fukui_plus_label1,pyridone_polarizability_label1,pyridone_dipole_label1,pyridone_p_int_label1,pyridone_vbur_label1,pyridone_B1_label1,pyridone_B5_label1,pyridone_L_label1,pyridone_charge_label2,pyridone_fukui_minus_label2,pyridone_fukui_zero_label2,pyridone_fukui_plus_label2,pyridone_polarizability_label2,pyridone_dipole_label2,pyridone_p_int_label2,pyridone_vbur_label2,pyridone_B1_label2,pyridone_B5_label2,pyridone_L_label2,pyridone_charge_label3,pyridone_fukui_minus_label3,pyridone_fukui_zero_label3,pyridone_fukui_plus_label3,pyridone_polarizability_label3,pyridone_dipole_label3,pyridone_p_int_label3,pyridone_vbur_label3,pyridone_B1_label3,pyridone_B5_label3,pyridone_L_label3
i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,0,-0.388311,-0.319427,8.5047,1.0886,239.526291,0.3509,0.485,0.543,0.601,32.986,0.875182,27.074261,39.668165,1.763298,5.084882,5.775934,-0.350701,0.042,0.023,0.003,6.028,0.185557,20.089336,48.406634,2.1,5.473037,5.822896,-0.146814,-0.005,0.011,0.026,7.303,0.075374,30.874694,64.818999,…,-0.380848,-0.250706,8.9152,-1.3769,-0.379884,0.112,0.091,0.07,6.107,0.198239,14.02238,28.841898,1.700434,3.285902,6.667683,0.32495,0.054,0.051,0.048,1.267,0.08328,11.822243,29.208209,1.52,6.096487,5.617428,-0.277363,0.195,0.178,0.16,7.847,0.278937,18.253291,45.716083,1.7,4.440932,4.6939
1,0,-0.384586,-0.317421,8.314,1.0756,239.128401,0.331844,0.458,0.523,0.588,33.036,0.901613,27.329587,39.952362,2.117552,5.077564,6.085845,-0.350211,0.043,0.023,0.002,6.027,0.185263,20.721648,49.129939,2.1,5.595524,6.066393,-0.160094,-0.014,0.005,0.025,7.346,0.09521,32.479729,70.157589,…,-0.380848,-0.250706,8.9152,-1.3769,-0.379884,0.112,0.091,0.07,6.107,0.198239,14.02238,28.841898,1.700434,3.285902,6.667683,0.32495,0.054,0.051,0.048,1.267,0.08328,11.822243,29.208209,1.52,6.096487,5.617428,-0.277363,0.195,0.178,0.16,7.847,0.278937,18.253291,45.716083,1.7,4.440932,4.6939
2,0,-0.381723,-0.314163,7.9751,1.0194,239.453323,0.334102,0.374,0.481,0.588,33.03,0.901504,27.795445,40.050783,2.252704,5.400756,7.650898,-0.351742,0.032,0.016,-0.0,6.031,0.184999,21.14598,49.525368,2.1,7.376626,5.946715,-0.156069,-0.02,0.002,0.025,7.333,0.07419,36.036076,70.994456,…,-0.380848,-0.250706,8.9152,-1.3769,-0.379884,0.112,0.091,0.07,6.107,0.198239,14.02238,28.841898,1.700434,3.285902,6.667683,0.32495,0.054,0.051,0.048,1.267,0.08328,11.822243,29.208209,1.52,6.096487,5.617428,-0.277363,0.195,0.178,0.16,7.847,0.278937,18.253291,45.716083,1.7,4.440932,4.6939
3,0,-0.381061,-0.315206,8.0633,1.0507,239.322728,0.329289,0.401,0.493,0.585,33.042,0.912037,27.824986,40.488143,2.399934,5.083051,6.572473,-0.353415,0.038,0.019,-0.001,6.035,0.185629,21.36554,50.069884,2.1,6.525664,5.779004,-0.152234,-0.009,0.009,0.027,7.321,0.086977,37.912943,74.040253,…,-0.380848,-0.250706,8.9152,-1.3769,-0.379884,0.112,

In [8]:
df_y = pl.read_excel("combined.xlsx").sort(["pyridone", "backbone"])
df_y = df_y[["pyridone", "backbone", "alpha_av", "beta_av"]]
df_y = df_y.with_columns((pl.col("beta_av") / pl.col("alpha_av")).log().alias("beta_alpha"))

In [9]:
df_y

pyridone,backbone,alpha_av,beta_av,beta_alpha
i64,i64,f64,f64,f64
0,0,4.114632,51.388357,2.524862
0,1,3.998281,51.539999,2.556494
0,2,3.428027,47.832328,2.635717
0,3,3.636433,47.845209,2.576968
0,4,3.323485,43.960253,2.582272
…,…,…,…,…
12,3,3.583822,46.114252,2.554692
12,4,3.742613,49.74711,2.587168
12,5,1.175172,48.959898,3.729587


In [10]:
df = df_X.join(df_y, on=["backbone", "pyridone"], how="left", maintain_order="left")

In [11]:
df

backbone,pyridone,backbone_homo,backbone_lumo,backbone_ip,backbone_ea,backbone_cone_angle,backbone_charge_label1,backbone_fukui_minus_label1,backbone_fukui_zero_label1,backbone_fukui_plus_label1,backbone_polarizability_label1,backbone_dipole_label1,backbone_p_int_label1,backbone_vbur_label1,backbone_B1_label1,backbone_B5_label1,backbone_L_label1,backbone_charge_label3,backbone_fukui_minus_label3,backbone_fukui_zero_label3,backbone_fukui_plus_label3,backbone_polarizability_label3,backbone_dipole_label3,backbone_p_int_label3,backbone_vbur_label3,backbone_B1_label3,backbone_B5_label3,backbone_L_label3,backbone_charge_label7,backbone_fukui_minus_label7,backbone_fukui_zero_label7,backbone_fukui_plus_label7,backbone_polarizability_label7,backbone_dipole_label7,backbone_p_int_label7,backbone_vbur_label7,…,pyridone_ea,pyridone_charge_label1,pyridone_fukui_minus_label1,pyridone_fukui_zero_label1,pyridone_fukui_plus_label1,pyridone_polarizability_label1,pyridone_dipole_label1,pyridone_p_int_label1,pyridone_vbur_label1,pyridone_B1_label1,pyridone_B5_label1,pyridone_L_label1,pyridone_charge_label2,pyridone_fukui_minus_label2,pyridone_fukui_zero_label2,pyridone_fukui_plus_label2,pyridone_polarizability_label2,pyridone_dipole_label2,pyridone_p_int_label2,pyridone_vbur_label2,pyridone_B1_label2,pyridone_B5_label2,pyridone_L_label2,pyridone_charge_label3,pyridone_fukui_minus_label3,pyridone_fukui_zero_label3,pyridone_fukui_plus_label3,pyridone_polarizability_label3,pyridone_dipole_label3,pyridone_p_int_label3,pyridone_vbur_label3,pyridone_B1_label3,pyridone_B5_label3,pyridone_L_label3,alpha_av,beta_av,beta_alpha
i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,0,-0.388311,-0.319427,8.5047,1.0886,239.526291,0.3509,0.485,0.543,0.601,32.986,0.875182,27.074261,39.668165,1.763298,5.084882,5.775934,-0.350701,0.042,0.023,0.003,6.028,0.185557,20.089336,48.406634,2.1,5.473037,5.822896,-0.146814,-0.005,0.011,0.026,7.303,0.075374,30.874694,64.818999,…,-1.3769,-0.379884,0.112,0.091,0.07,6.107,0.198239,14.02238,28.841898,1.700434,3.285902,6.667683,0.32495,0.054,0.051,0.048,1.267,0.08328,11.822243,29.208209,1.52,6.096487,5.617428,-0.277363,0.195,0.178,0.16,7.847,0.278937,18.253291,45.716083,1.7,4.440932,4.6939,4.114632,51.388357,2.524862
1,0,-0.384586,-0.317421,8.314,1.0756,239.128401,0.331844,0.458,0.523,0.588,33.036,0.901613,27.329587,39.952362,2.117552,5.077564,6.085845,-0.350211,0.043,0.023,0.002,6.027,0.185263,20.721648,49.129939,2.1,5.595524,6.066393,-0.160094,-0.014,0.005,0.025,7.346,0.09521,32.479729,70.157589,…,-1.3769,-0.379884,0.112,0.091,0.07,6.107,0.198239,14.02238,28.841898,1.700434,3.285902,6.667683,0.32495,0.054,0.051,0.048,1.267,0.08328,11.822243,29.208209,1.52,6.096487,5.617428,-0.277363,0.195,0.178,0.16,7.847,0.278937,18.253291,45.716083,1.7,4.440932,4.6939,3.998281,51.539999,2.556494
2,0,-0.381723,-0.314163,7.9751,1.0194,239.453323,0.334102,0.374,0.481,0.588,33.03,0.901504,27.795445,40.050783,2.252704,5.400756,7.650898,-0.351742,0.032,0.016,-0.0,6.031,0.184999,21.14598,49.525368,2.1,7.376626,5.946715,-0.156069,-0.02,0.002,0.025,7.333,0.07419,36.036076,70.994456,…,-1.3769,-0.379884,0.112,0.091,0.07,6.107,0.198239,14.02238,28.841898,1.700434,3.285902,6.667683,0.32495,0.054,0.051,0.048,1.267,0.08328,11.822243,29.208209,1.52,6.096487,5.617428,-0.277363,0.195,0.178,0.16,7.847,0.278937,18.253291,45.716083,1.7,4.440932,4.6939,3.428027,47.832328,2.635717
3,0,-0.381061,-0.315206,8.0633,1.0507,239.322728,0.329289,0.401,0.493,0.585,33.042,0.912037,27.824986,40.488143,2.399934,5.083051,6.572473,-0.353415,0.038,0.019,-0.001,6.035,0.185629,21.36554,50.069884,2.1,6.525664,5.779004,-0.152234,-0.009,0.009,0.027,7.321,0.086977,37.912943,74.040253,…,-1.3769,-0.379884,0.112,0.091,0.07,6.107,0.198239,14.02238,2

In [12]:
df.write_parquet("dataset.parquet")